Question: is it worth the effort of deriving and deploying a custom historical weighting scheme? Or does a simple k-NN recommender suffice?

Part 4: A/B Testing.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

The following cell is copied from streamlit/app.py, and just initialises the functions.

In [2]:
DATA = Path("../data")
D_lyrics_df = pd.read_csv(DATA / 'lyrics_dissimilarities.csv')
D_audio_df = pd.read_csv(DATA / 'audio_dissimilarities.csv')
name_to_id = pd.read_csv(DATA / 'name_to_id.csv').set_index('name')
id_to_name = name_to_id.reset_index().set_index("id")

# cast index/column names to numbers
D_audio_df.index = D_audio_df.index.astype(int)
D_audio_df.columns = D_audio_df.columns.astype(int)

D_lyrics_df.index = D_lyrics_df.index.astype(int)
D_lyrics_df.columns = D_lyrics_df.columns.astype(int)

# recommendation algorithms - k-NN based
def recommend(query_name, alpha=0.5, k=5):
    D = alpha * D_audio_df + (1 - alpha) * D_lyrics_df
    query_id = name_to_id.loc[query_name, 'id']
    distances = D.loc[query_id].copy()
    distances.loc[query_id] = np.inf  # don't recommend itself
    return [id_to_name.loc[x]['name'] for x in distances.nsmallest(k).index]

def recommend_weighted(query_name: str, prev_queries: list, alpha=0.5, k=5):
    """Takes query and a list of previous queries (ordered, most recent first)"""

    D = alpha * D_audio_df + (1 - alpha) * D_lyrics_df

    # weight function for current query
    if prev_queries:
        s_current = 0.8
    else:
        s_current = 1

    query_id = name_to_id.loc[query_name, 'id']
    distances = D.loc[query_id].copy()*s_current
    distances.loc[query_id] = np.inf  # don't recommend itself

    # define weight function for previous queries
    def W(i, S):
        return (2*(S-i))/(5*S*(S+1))
        
    for i, query in enumerate(prev_queries):
        query_id = name_to_id.loc[query, 'id']
        query_distances = D.loc[query_id].copy() * W(i, len(prev_queries))
        query_distances.loc[query_id] += 1/(i+1) # make it less likely to recommend something already searched 
        distances = distances + query_distances
    return [id_to_name.loc[x]['name'] for x in distances.nsmallest(k).index]

def search(query_name: str, prev_queries: list, balance='equal', use_prev=True, k=5):
    """
    Returns top 5 recommendations given query name and list of previous queries. 
    balance takes values in {'equal', 'lyrics', 'audio'}, changing the fusion weights of the dissimilarity matrices.
    use_prev is boolean. True means previous searches are used to enhance recommendations.
    """

    balance_to_alpha = {
        'equal': 0.5,
        'lyrics': 0.3, 
        'audio': 0.7
    }

    assert type(use_prev) == bool
    if use_prev == True:
        return recommend_weighted(query_name, prev_queries, alpha=balance_to_alpha[balance], k=k)
    else:
        return recommend(query_name, alpha=balance_to_alpha[balance], k=k)

Since this app is not live, user behaviour has to be simulated.

Users will be partitioned into three categories by the albums that they listen to.

Category 1 (country/rock):
* Taylor Swift
* Fearless
* Speak Now
* Red

Category 2 (pop):
* 1989
* Reputation
* Lover
* Midnights

Category 3 (poetic/indie):
* folklore
* evermore
* TTPD

Users will listen to three distinct songs from their category. If three or more songs from five recommendations are in their category and weren't in the history, they will 'like' the recommendations. We test both recommendation systems (with and without weights).

Experimental hypotheses:
* $H_0$: The recommendation systems perform the same.
* $H_1$: The weighted recommendation system performs better.

Even though $H_1$ is directional, we cannot rule out the possibility that the weighted recommendation system is worse. So a two-tailed test is used.

Simulate the baseline with 60 users (20 from each category).

In [3]:
merged = pd.read_csv(DATA / 'merged.csv')
album_lookup = pd.DataFrame({'name': merged['audio_name'], 'album': merged['album']})
album_lookup['album'].unique()

array(['THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY',
       "1989 (Taylor's Version) [Deluxe]", "Speak Now (Taylor's Version)",
       'Midnights (The Til Dawn Edition)', "Red (Taylor's Version)",
       "Fearless (Taylor's Version)", 'evermore (deluxe version)',
       'folklore (deluxe version)', 'Lover', 'reputation',
       'Taylor Swift (Deluxe Edition)'], dtype=object)

In [4]:
cat_1 = list(album_lookup[album_lookup['album'].isin([
    'Taylor Swift (Deluxe Edition)', 
    "Fearless (Taylor's Version)", 
    "Speak Now (Taylor's Version)",
    "Red (Taylor's Version)"
    ])]['name'])

cat_2 = list(album_lookup[album_lookup['album'].isin([
    "1989 (Taylor's Version) [Deluxe]", 
    "reputation", 
    'Lover',
    'Midnights (The Til Dawn Edition)'
    ])]['name'])

cat_3 = list(album_lookup[album_lookup['album'].isin([
    'folklore (deluxe version)', 
    'evermore (deluxe version)', 
    'THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY'
    ])]['name'])

In [5]:
import random
random.seed(42)
likes = 0

# for songs [A, B, C], A was listened to most recently, followed by B and C.
for cat in [cat_1, cat_2, cat_3]:
    for users in range(20):
        count = 0
        history = random.sample(cat, 3)
        recs = search(history[0], history[1:], use_prev=False, k=5)
        for rec in recs:
            if rec in cat and rec not in history:
                count += 1
        if count >= 3:
            likes += 1

likes/60

0.7

A change from 0.7 to 0.75 in like-rate is worth deploying.

For significance level $\alpha$, power, $p_1$ baseline, $p_2$ improved proportion, and $\bar{p} = \frac{p_1 + p_2}{2}$, the required sample size $n$ is given by $$n = \frac{\left(z_{\alpha/2}\sqrt{2\bar{p}(1-\bar{p})} + z_{\beta}\sqrt{p_1(1-p_1) + p_2(1-p_2)}\right)^2}{(p_1 - p_2)^2}.$$

This can be calculated using statsmodels.


In [6]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

p1 = 0.7
p2 = 0.75

effect_size = proportion_effectsize(p1, p2)  # Cohen's h

analysis = NormalIndPower()
n_per_arm = analysis.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.8,
    ratio=1.0
)

'Required sample size per arm: {n_per_arm:.0f}'

'Required sample size per arm: {n_per_arm:.0f}'

1250 per arm means 417 users per category, 2502 users in total. 

Stratification is simple in this case because the exact number of simulations can be chosen. In real-life, this would be a random sample of 417 users per category.

We simulate using 417 trials for each category in each arm.

In [7]:
# Control arm
likes_per_category = []

# for songs [A, B, C], A was listened to most recently, followed by B and C.
for cat in [cat_1, cat_2, cat_3]:
    likes = 0
    for users in range(417):
        count = 0
        history = random.sample(cat, 3)
        recs = search(history[0], history[1:], use_prev=False, k=5)
        for rec in recs:
            if rec in cat and rec not in history:
                count += 1
        if count >= 3:
            likes += 1

    likes_per_category.append(likes)

likes_per_category, sum(likes_per_category)

([365, 218, 309], 892)

In [8]:
# alternative arm
likes_per_category = []

# for songs [A, B, C], A was listened to most recently, followed by B and C.
for cat in [cat_1, cat_2, cat_3]:
    likes = 0
    for users in range(417):
        count = 0
        history = random.sample(cat, 3)
        recs = search(history[0], history[1:], use_prev=True, k=5)
        for rec in recs:
            if rec in cat and rec not in history:
                count += 1
        if count >= 3:
            likes += 1

    likes_per_category.append(likes)

likes_per_category, sum(likes_per_category)

([394, 254, 357], 1005)

In [9]:
from statsmodels.stats.proportion import proportions_ztest

control_likes, control_n = 892, 1251
alternative_likes, alternative_n = 1005, 1251

count = np.array([alternative_likes, control_likes])
n_obs = np.array([alternative_n, control_n])

z_stat, p_value = proportions_ztest(count, n_obs, alternative='two-sided')

f"z = {z_stat:.3f}, p = {p_value}"

'z = 5.276, p = 1.3198198900843394e-07'

$p < 0.05$ (p = $1.32 \times 10^{-7}$) so the result is significant overall and $H_0$ should be rejected.

We can also check by category.

In [10]:
control_likes_per_category = [365, 218, 309]
alternative_likes_per_category = [394, 254, 357]

for i in range(3):
    control_likes, control_n = control_likes_per_category[i], 417
    alternative_likes, alternative_n = alternative_likes_per_category[i], 417

    count = np.array([alternative_likes, control_likes])
    n_obs = np.array([alternative_n, control_n])

    z_stat, p_value = proportions_ztest(count, n_obs, alternative='two-sided')

    print(f"For category {i+1}, z = {z_stat:.3f}, p = {p_value:.7f}")

For category 1, z = 3.510, p = 0.0004478
For category 2, z = 2.515, p = 0.0118989
For category 3, z = 4.144, p = 0.0000341


The result is significant for all categories. However, we have to account for multiple hypothesis testing. Bonferroni's correction can be used (multiply $p$-values by number of tests and compare to significance level).

In [11]:
p_values = np.array([1.32e-7, 4.48e-4, 1.19e-2, 3.41e-5])
results = pd.DataFrame({
    'subset': ['All', 'Category 1', 'Category 2', 'Category 3'],
    'p_value': p_values,
    'adjusted_p_value': p_values*4
}).set_index('subset')

results['significant'] = results['adjusted_p_value']<0.05

results

,p_value,adjusted_p_value,significant
subset,,,
All,1.320000e-07,5.280000e-07,True
Category 1,4.480000e-04,1.792000e-03,True
Category 2,1.190000e-02,4.760000e-02,True
Category 3,3.410000e-05,1.364000e-04,True


All $p$-values were significant even after Bonferroni's correction. However, the adjusted $p$-value for Category 2 is quite close to 0.05. With borderline results like this, peeking can have an effect on the outcome of the test.